In [1]:
import pandas as pd
import numpy as np

In [4]:
features_df = pd.read_csv('data/features_processed.csv')
features_df.head()

,date,home_team,away_team,home_GF5,home_GA5,home_W5,home_D5,home_L5,home_points5,home_mean_competition5,...,home_unbeaten_streak,away_unbeaten_streak,h2h_home_wins,h2h_away_wins,h2h_draws,h2h_matches,neutral,competition_level,home_score,away_score
0,2000-01-04,Egypt,Togo,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,2,1
1,2000-01-07,Tunisia,Togo,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,7,0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,1,0,0
3,2000-01-09,Mexico,Iran,0.0,0.0,0,0,0,0,0.0,...,0,0,0,0,0,0,1,1,2,1
4,2000-01-09,Ivory Coast,Egypt,0.0,0.0,0,0,0,0,0.0,...,0,1,0,0,0,0,0,1,2,0


In [5]:
features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25343 entries, 0 to 25342
Data columns (total 44 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     25343 non-null  str    
 1   home_team                25343 non-null  str    
 2   away_team                25343 non-null  str    
 3   home_GF5                 25343 non-null  float64
 4   home_GA5                 25343 non-null  float64
 5   home_W5                  25343 non-null  int64  
 6   home_D5                  25343 non-null  int64  
 7   home_L5                  25343 non-null  int64  
 8   home_points5             25343 non-null  int64  
 9   home_mean_competition5   25343 non-null  float64
 10  home_matches_used5       25343 non-null  int64  
 11  away_GF5                 25343 non-null  float64
 12  away_GA5                 25343 non-null  float64
 13  away_W5                  25343 non-null  int64  
 14  away_D5                  25343 no

In [6]:
features_df["date"] = pd.to_datetime(features_df["date"])

In [9]:
features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25343 entries, 0 to 25342
Data columns (total 44 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     25343 non-null  datetime64[us]
 1   home_team                25343 non-null  str           
 2   away_team                25343 non-null  str           
 3   home_GF5                 25343 non-null  float64       
 4   home_GA5                 25343 non-null  float64       
 5   home_W5                  25343 non-null  int64         
 6   home_D5                  25343 non-null  int64         
 7   home_L5                  25343 non-null  int64         
 8   home_points5             25343 non-null  int64         
 9   home_mean_competition5   25343 non-null  float64       
 10  home_matches_used5       25343 non-null  int64         
 11  away_GF5                 25343 non-null  float64       
 12  away_GA5                 25343 non-null  fl

In [10]:
X = features_df.drop(
    columns=[
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
)

y_home = features_df["home_score"]
y_away = features_df["away_score"]

In [11]:
train_mask = features_df["date"] < "2023-01-01"

val_mask = (
    (features_df["date"] >= "2023-01-01")
    &
    (features_df["date"] < "2025-01-01")
)

test_mask = features_df["date"] >= "2025-01-01"

In [12]:
X_train = X[train_mask]
X_val = X[val_mask]
X_test = X[test_mask]

In [13]:
y_home_train = y_home[train_mask]
y_home_val = y_home[val_mask]
y_home_test = y_home[test_mask]

In [14]:
y_away_train = y_away[train_mask]
y_away_val = y_away[val_mask]
y_away_test = y_away[test_mask]

In [15]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(21745, 39)
(2285, 39)
(1313, 39)


In [16]:
print(y_home_train.shape)
print(y_home_val.shape)
print(y_home_test.shape)

(21745,)
(2285,)
(1313,)


In [18]:
from xgboost import XGBRegressor

In [19]:
home_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

home_model.fit(
    X_train,
    y_home_train
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [20]:
away_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

away_model.fit(
    X_train,
    y_away_train
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [21]:
home_pred_val = home_model.predict(X_val)
away_pred_val = away_model.predict(X_val)

In [22]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [23]:

mae_home = mean_absolute_error(y_home_val, home_pred_val)
mae_away = mean_absolute_error(y_away_val, away_pred_val)

rmse_home = root_mean_squared_error(y_home_val, home_pred_val)
rmse_away = root_mean_squared_error(y_away_val, away_pred_val)

print(mae_home)
print(mae_away)

print(rmse_home)
print(rmse_away)

1.0374577045440674
0.8584217429161072
1.389655590057373
1.1219844818115234


In [24]:
home_pred_round = np.round(home_pred_val).astype(int)
away_pred_round = np.round(away_pred_val).astype(int)

In [25]:
results = pd.DataFrame({
    "home_real": y_home_val,
    "home_pred": home_pred_round,
    "away_real": y_away_val,
    "away_pred": away_pred_round
})

results.head(20)

,home_real,home_pred,away_real,away_pred
21745,3,4,1,0
21746,1,1,2,1
21747,3,3,0,0
21748,4,1,1,1
21749,0,0,2,2
21750,0,1,0,1
21751,0,1,0,1
21752,2,1,1,1
21753,0,1,2,1
21754,1,1,0,1


In [27]:
def outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

In [28]:
real_outcome = [
    outcome(h, a)
    for h, a in zip(
        results["home_real"],
        results["away_real"]
    )
]

In [29]:
pred_outcome = [
    outcome(h, a)
    for h, a in zip(
        results["home_pred"],
        results["away_pred"]
    )
]

In [30]:
from sklearn.metrics import accuracy_score

In [31]:
accuracy_score(real_outcome, pred_outcome)

0.5540481400437637

In [32]:
exact_score = (
    (results["home_real"] == results["home_pred"])
    &
    (results["away_real"] == results["away_pred"])
).mean()

print(exact_score)

0.11203501094091904


### importancia de features

In [33]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": home_model.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
28,elo_diff,0.085838
22,away_GA10,0.073998
34,h2h_away_wins,0.048099
9,away_GA5,0.047014
25,away_matches_used10,0.045968
15,away_matches_used5,0.039269
37,neutral,0.033418
26,home_elo,0.032411
20,home_matches_used10,0.028499
38,competition_level,0.028248


In [34]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": away_model.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
28,elo_diff,0.087952
17,home_GA10,0.087534
1,home_GA5,0.056045
37,neutral,0.055696
38,competition_level,0.031102
25,away_matches_used10,0.029613
23,away_points10,0.029588
33,h2h_home_wins,0.029159
9,away_GA5,0.027451
22,away_GA10,0.026959


## Modelo 2

### Eliminar
[
'h2h_home_wins',
'h2h_away_wins',
'h2h_draws',
'h2h_matches'
]

In [36]:
features_model2 = features_df.copy()

features_model2["date"] = pd.to_datetime(features_model2["date"])

In [37]:
h2h_cols = [
    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_matches"
]

In [38]:
X_model2 = features_model2.drop(
    columns=[
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ] + h2h_cols
)

y_home = features_model2["home_score"]
y_away = features_model2["away_score"]

In [39]:
train_mask = features_model2["date"] < "2023-01-01"

val_mask = (
    (features_model2["date"] >= "2023-01-01") &
    (features_model2["date"] < "2025-01-01")
)

test_mask = features_model2["date"] >= "2025-01-01"

In [40]:
X_train_m2 = X_model2[train_mask]
X_val_m2 = X_model2[val_mask]
X_test_m2 = X_model2[test_mask]

y_home_train = y_home[train_mask]
y_home_val = y_home[val_mask]
y_home_test = y_home[test_mask]

y_away_train = y_away[train_mask]
y_away_val = y_away[val_mask]
y_away_test = y_away[test_mask]

In [41]:
home_model_m2 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

away_model_m2 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

home_model_m2.fit(X_train_m2, y_home_train)
away_model_m2.fit(X_train_m2, y_away_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [42]:
home_pred_val_m2 = home_model_m2.predict(X_val_m2)
away_pred_val_m2 = away_model_m2.predict(X_val_m2)

In [43]:
mae_home_m2 = mean_absolute_error(y_home_val, home_pred_val_m2)
mae_away_m2 = mean_absolute_error(y_away_val, away_pred_val_m2)

rmse_home_m2 = root_mean_squared_error(y_home_val, home_pred_val_m2)
rmse_away_m2 = root_mean_squared_error(y_away_val, away_pred_val_m2)

print("Modelo 2 - Sin H2H")
print("MAE Home:", mae_home_m2)
print("MAE Away:", mae_away_m2)
print("RMSE Home:", rmse_home_m2)
print("RMSE Away:", rmse_away_m2)

Modelo 2 - Sin H2H
MAE Home: 1.0490875244140625
MAE Away: 0.8652929663658142
RMSE Home: 1.401103138923645
RMSE Away: 1.1270060539245605


In [44]:
home_pred_round_m2 = np.round(home_pred_val_m2).astype(int)
away_pred_round_m2 = np.round(away_pred_val_m2).astype(int)

home_pred_round_m2 = np.clip(home_pred_round_m2, 0, None)
away_pred_round_m2 = np.clip(away_pred_round_m2, 0, None)

In [45]:
results_m2 = pd.DataFrame({
    "home_real": y_home_val.values,
    "away_real": y_away_val.values,
    "home_pred": home_pred_round_m2,
    "away_pred": away_pred_round_m2
})

In [46]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

In [47]:
results_m2["real_outcome"] = results_m2.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_m2["pred_outcome"] = results_m2.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

In [48]:
accuracy_wdl_m2 = accuracy_score(
    results_m2["real_outcome"],
    results_m2["pred_outcome"]
)

exact_score_m2 = (
    (results_m2["home_real"] == results_m2["home_pred"]) &
    (results_m2["away_real"] == results_m2["away_pred"])
).mean()

print("Accuracy W-D-L:", accuracy_wdl_m2)
print("Exact Score Accuracy:", exact_score_m2)

Accuracy W-D-L: 0.5531728665207878
Exact Score Accuracy: 0.10547045951859957


In [49]:
comparison = pd.DataFrame({
    "model": ["Modelo 1 - Completo", "Modelo 2 - Sin H2H"],
    "MAE_home": [1.0374577045440674, mae_home_m2],
    "MAE_away": [0.8584217429161072, mae_away_m2],
    "RMSE_home": [1.389655590057373, rmse_home_m2],
    "RMSE_away": [1.1219844818115234, rmse_away_m2],
    "Accuracy_WDL": [0.5540481400437637, accuracy_wdl_m2],
    "Exact_score": [0.11203501094091904, exact_score_m2]
})

comparison

,model,MAE_home,MAE_away,RMSE_home,RMSE_away,Accuracy_WDL,Exact_score
0,Modelo 1 - Completo,1.037458,0.858422,1.389656,1.121984,0.554048,0.112035
1,Modelo 2 - Sin H2H,1.049088,0.865293,1.401103,1.127006,0.553173,0.105470


## Modelo 3
### Eliminar
[
'home_matches_used5',
'away_matches_used5',
'home_matches_used10',
'away_matches_used10'
]

In [50]:
features_model3 = features_df.copy()

features_model3["date"] = pd.to_datetime(features_model3["date"])

In [51]:
used_cols = [
'home_matches_used5',
'away_matches_used5',
'home_matches_used10',
'away_matches_used10'
]

In [52]:
X_model3 = features_model3.drop(
    columns=[
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ] + used_cols
)

y_home = features_model3["home_score"]
y_away = features_model3["away_score"]

In [53]:
train_mask = features_model3["date"] < "2023-01-01"

val_mask = (
    (features_model3["date"] >= "2023-01-01") &
    (features_model3["date"] < "2025-01-01")
)

test_mask = features_model3["date"] >= "2025-01-01"

In [54]:
X_train_m3 = X_model3[train_mask]
X_val_m3 = X_model3[val_mask]
X_test_m3 = X_model3[test_mask]

y_home_train = y_home[train_mask]
y_home_val = y_home[val_mask]
y_home_test = y_home[test_mask]

y_away_train = y_away[train_mask]
y_away_val = y_away[val_mask]
y_away_test = y_away[test_mask]

In [55]:
home_model_m3 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

away_model_m3 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

home_model_m3.fit(X_train_m3, y_home_train)
away_model_m3.fit(X_train_m3, y_away_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [56]:
home_pred_val_m3 = home_model_m3.predict(X_val_m3)
away_pred_val_m3 = away_model_m3.predict(X_val_m3)

In [57]:
mae_home_m3 = mean_absolute_error(y_home_val, home_pred_val_m3)
mae_away_m3 = mean_absolute_error(y_away_val, away_pred_val_m3)

rmse_home_m3 = root_mean_squared_error(y_home_val, home_pred_val_m3)
rmse_away_m3 = root_mean_squared_error(y_away_val, away_pred_val_m3)

print("Modelo 2 - Sin H2H")
print("MAE Home:", mae_home_m3)
print("MAE Away:", mae_away_m3)
print("RMSE Home:", rmse_home_m3)
print("RMSE Away:", rmse_away_m3)

Modelo 2 - Sin H2H
MAE Home: 1.038977026939392
MAE Away: 0.8594841957092285
RMSE Home: 1.3877151012420654
RMSE Away: 1.1249585151672363


In [58]:
home_pred_round_m3 = np.round(home_pred_val_m3).astype(int)
away_pred_round_m3 = np.round(away_pred_val_m3).astype(int)

home_pred_round_m3 = np.clip(home_pred_round_m3, 0, None)
away_pred_round_m3 = np.clip(away_pred_round_m3, 0, None)

In [59]:
results_m3 = pd.DataFrame({
    "home_real": y_home_val.values,
    "away_real": y_away_val.values,
    "home_pred": home_pred_round_m3,
    "away_pred": away_pred_round_m3
})

In [60]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

In [61]:
results_m3["real_outcome"] = results_m3.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_m3["pred_outcome"] = results_m3.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

In [62]:
accuracy_wdl_m3 = accuracy_score(
    results_m3["real_outcome"],
    results_m3["pred_outcome"]
)

exact_score_m3 = (
    (results_m3["home_real"] == results_m3["home_pred"]) &
    (results_m3["away_real"] == results_m3["away_pred"])
).mean()

print("Accuracy W-D-L:", accuracy_wdl_m3)
print("Exact Score Accuracy:", exact_score_m3)

Accuracy W-D-L: 0.5483588621444201
Exact Score Accuracy: 0.10940919037199125


In [63]:
comparison = pd.DataFrame({
    "model": ["Modelo 1 - Completo", "Modelo 2 - Sin H2H", "Modelo 3 - Sin últimos partidos"],
    "MAE_home": [1.0374577045440674, mae_home_m2, mae_home_m3],
    "MAE_away": [0.8584217429161072, mae_away_m2, mae_away_m3],
    "RMSE_home": [1.389655590057373, rmse_home_m2, rmse_home_m3],
    "RMSE_away": [1.1219844818115234, rmse_away_m2, rmse_away_m3],
    "Accuracy_WDL": [0.5540481400437637, accuracy_wdl_m2, accuracy_wdl_m3],
    "Exact_score": [0.11203501094091904, exact_score_m2, exact_score_m3]
})

comparison

,model,MAE_home,MAE_away,RMSE_home,RMSE_away,Accuracy_WDL,Exact_score
0,Modelo 1 - Completo,1.037458,0.858422,1.389656,1.121984,0.554048,0.112035
1,Modelo 2 - Sin H2H,1.049088,0.865293,1.401103,1.127006,0.553173,0.105470
2,Modelo 3 - Sin últimos partidos,1.038977,0.859484,1.387715,1.124959,0.548359,0.109409


## Modelo 4 
### Solo variables importantes

In [64]:
features_model4 = features_df.copy()
features_model4["date"] = pd.to_datetime(features_model4["date"])

In [65]:
selected_features_m4 = [
    # Últimos 10 partidos
    "home_GF10",
    "home_GA10",
    "home_points10",
    "home_mean_competition10",

    "away_GF10",
    "away_GA10",
    "away_points10",
    "away_mean_competition10",

    # Elo
    "home_elo",
    "away_elo",
    "elo_diff",

    # H2H
    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_matches",

    # Contexto
    "neutral",
    "competition_level"
]

In [66]:
X_model4 = features_model4[selected_features_m4]

y_home = features_model4["home_score"]
y_away = features_model4["away_score"]

In [67]:
train_mask = features_model4["date"] < "2023-01-01"

val_mask = (
    (features_model4["date"] >= "2023-01-01") &
    (features_model4["date"] < "2025-01-01")
)

test_mask = features_model4["date"] >= "2025-01-01"

In [68]:
X_train_m4 = X_model4[train_mask]
X_val_m4 = X_model4[val_mask]
X_test_m4 = X_model4[test_mask]

y_home_train = y_home[train_mask]
y_home_val = y_home[val_mask]
y_home_test = y_home[test_mask]

y_away_train = y_away[train_mask]
y_away_val = y_away[val_mask]
y_away_test = y_away[test_mask]

In [69]:
home_model_m4 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

away_model_m4 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

home_model_m4.fit(X_train_m4, y_home_train)
away_model_m4.fit(X_train_m4, y_away_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [70]:
home_pred_val_m4 = home_model_m4.predict(X_val_m4)
away_pred_val_m4 = away_model_m4.predict(X_val_m4)

In [71]:
mae_home_m4 = mean_absolute_error(y_home_val, home_pred_val_m4)
mae_away_m4 = mean_absolute_error(y_away_val, away_pred_val_m4)

rmse_home_m4 = root_mean_squared_error(y_home_val, home_pred_val_m4)
rmse_away_m4 = root_mean_squared_error(y_away_val, away_pred_val_m4)

print("Modelo 4 - Variables fuertes")
print("MAE Home:", mae_home_m4)
print("MAE Away:", mae_away_m4)
print("RMSE Home:", rmse_home_m4)
print("RMSE Away:", rmse_away_m4)

Modelo 4 - Variables fuertes
MAE Home: 1.0409611463546753
MAE Away: 0.8608290553092957
RMSE Home: 1.3992283344268799
RMSE Away: 1.1257758140563965


In [72]:
home_pred_round_m4 = np.round(home_pred_val_m4).astype(int)
away_pred_round_m4 = np.round(away_pred_val_m4).astype(int)

home_pred_round_m4 = np.clip(home_pred_round_m4, 0, None)
away_pred_round_m4 = np.clip(away_pred_round_m4, 0, None)

In [73]:
results_m4 = pd.DataFrame({
    "home_real": y_home_val.values,
    "away_real": y_away_val.values,
    "home_pred": home_pred_round_m4,
    "away_pred": away_pred_round_m4
})

In [74]:
def get_outcome(home, away):
    if home > away:
        return "H"
    elif home < away:
        return "A"
    else:
        return "D"

In [75]:
results_m4["real_outcome"] = results_m4.apply(
    lambda row: get_outcome(row["home_real"], row["away_real"]),
    axis=1
)

results_m4["pred_outcome"] = results_m4.apply(
    lambda row: get_outcome(row["home_pred"], row["away_pred"]),
    axis=1
)

In [76]:
accuracy_wdl_m4 = accuracy_score(
    results_m4["real_outcome"],
    results_m4["pred_outcome"]
)

exact_score_m4 = (
    (results_m4["home_real"] == results_m4["home_pred"]) &
    (results_m4["away_real"] == results_m4["away_pred"])
).mean()

print("Accuracy W-D-L:", accuracy_wdl_m4)
print("Exact Score Accuracy:", exact_score_m4)

Accuracy W-D-L: 0.5527352297592998
Exact Score Accuracy: 0.10897155361050329


In [77]:
comparison = pd.DataFrame({
    "model": [
        "Modelo 1 - Completo",
        "Modelo 2 - Sin H2H",
        "Modelo 3 - Sin matches_used",
        "Modelo 4 - Variables fuertes"
    ],
    "MAE_home": [
        1.0374577045440674,
        1.049088,
        1.038977,
        mae_home_m4
    ],
    "MAE_away": [
        0.8584217429161072,
        0.865293,
        0.859484,
        mae_away_m4
    ],
    "RMSE_home": [
        1.389655590057373,
        1.401103,
        1.387715,
        rmse_home_m4
    ],
    "RMSE_away": [
        1.1219844818115234,
        1.127006,
        1.124959,
        rmse_away_m4
    ],
    "Accuracy_WDL": [
        0.5540481400437637,
        0.553173,
        0.548359,
        accuracy_wdl_m4
    ],
    "Exact_score": [
        0.11203501094091904,
        0.105470,
        0.109409,
        exact_score_m4
    ]
})

comparison

,model,MAE_home,MAE_away,RMSE_home,RMSE_away,Accuracy_WDL,Exact_score
0,Modelo 1 - Completo,1.037458,0.858422,1.389656,1.121984,0.554048,0.112035
1,Modelo 2 - Sin H2H,1.049088,0.865293,1.401103,1.127006,0.553173,0.105470
2,Modelo 3 - Sin matches_used,1.038977,0.859484,1.387715,1.124959,0.548359,0.109409
3,Modelo 4 - Variables fuertes,1.040961,0.860829,1.399228,1.125776,0.552735,0.108972
